# Barred star-forming dwarfs: candidate vetting, MUSE coverage, geometry

Companion notebook to the pipeline in `scripts/`. It takes you from the
candidate list produced by steps 1-2 to a vetted, MUSE-covered, TW-ready
target list, interactively:

1. load and look at the candidates (`products/barred_dwarf_candidates.csv`)
2. **visual vetting** on Legacy Surveys cutouts -> `products/vetting.csv`
3. **ESO archive query** for MUSE cubes / raw frames (step 3)
4. optional: Galaxy Zoo DESI bars for the non-SGA dwarfs (step 2b)
5. **disc/bar geometry** from Legacy Surveys images and TW suitability (step 4)
6. TW demo on a synthetic disc, so you know what the real diagram should look like

Run from the repo root (`jupyter lab notebooks/`). Sections 2-5 need internet
access; everything else works offline. Re-running a section is safe: results
are written to `products/` and `data/` and re-loaded if present.

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "bartw").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML, Image
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from bartw.config import load_config, data_dir
cfg = load_config()
PROD = ROOT / "products"; PROD.mkdir(exist_ok=True)
print("repo:", ROOT)

## 1. The candidates

Produced by `scripts/01_select_desi_sf_dwarfs.py` + `scripts/02_crossmatch_bars.py`
from the paper's 14,786 star-forming dwarfs and the SGA bar catalogue.
`PRIORITY` 1-2 = z < 0.012 (bar resolvable with MUSE), 3 = z < 0.02, 4 = rest.

In [ ]:
cand = pd.read_csv(PROD / "barred_dwarf_candidates.csv")
print(len(cand), "candidates;", (cand.PRIORITY <= 2).sum(), "priority 1-2")
display(cand.groupby("PRIORITY").agg(n=("TARGETID", "size"), z_med=("Z", "median"),
        logM_med=("logmstar_h0678", "median"), incl_med=("INCL_DEG", "median"),
        bar_ellip_med=("BAR_ELLIP", "median")).round(3))

fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
ax[0].hist(cand.Z, bins=20); ax[0].axvline(cfg["desi"]["z_max"], color="C3", ls="--"); ax[0].set_xlabel("z")
ax[1].hist(cand.logmstar_h0678, bins=20); ax[1].set_xlabel("log M* (h=0.678)")
ax[2].hist(cand.INCL_DEG, bins=20); [ax[2].axvline(v, color="C3", ls="--") for v in (25, 70)]; ax[2].set_xlabel("inclination [deg]")
ax[3].scatter(1 - cand.DISC_BA, cand.BAR_ELLIP, c=cand.PRIORITY, cmap="viridis_r", s=15)
ax[3].plot([0, 1], [0.1, 1.1], "k:", lw=1); ax[3].set_xlabel("disc ellipticity"); ax[3].set_ylabel("bar ellipticity")
plt.tight_layout()

## 2. Visual vetting on Legacy Surveys images

The bar catalogue records a bar *measurement* for every entry, not a detection,
so each priority target must be checked by eye before it goes to the archive
query or into a proposal. The grid below shows DR10 grz colour cutouts
(1 arcmin on a side, north up, east left) with the SGA R25 and the adopted
disc PA drawn. Fill the `VET` dictionary in the next cell:

* `y` = clear bar, `w` = weak/uncertain bar, `n` = no bar, `x` = not a dwarf disc
  (edge-on, merger, companion inside the field, artefact).

Classifications are saved to `products/vetting.csv` and merged back below.

In [ ]:
import requests, io
from PIL import Image as PILImage

def cutout_jpg(ra, dec, size_arcsec=60, layer="ls-dr10", pixscale=0.262):
    size = int(size_arcsec / pixscale)
    url = (f"https://www.legacysurvey.org/viewer/jpeg-cutout?ra={ra}&dec={dec}"
           f"&layer={layer}&pixscale={pixscale}&size={size}")
    r = requests.get(url, timeout=60); r.raise_for_status()
    return PILImage.open(io.BytesIO(r.content)), size

def viewer_link(ra, dec):
    return f"https://www.legacysurvey.org/viewer?ra={ra:.5f}&dec={dec:.5f}&layer=ls-dr10&zoom=14"

def show_grid(df, ncol=5, size_arcsec=60, pixscale=0.262):
    n = len(df); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.4 * nrow))
    for ax, (_, r) in zip(np.atleast_1d(axes).ravel(), df.iterrows()):
        try:
            img, size = cutout_jpg(r.RA, r.DEC, size_arcsec, pixscale=pixscale)
            ax.imshow(img, origin="upper")
            c = size / 2
            if np.isfinite(r.sma_sb25) and r.sma_sb25 > 0:   # R25 ellipse (SGA): sma in arcsec
                from matplotlib.patches import Ellipse
                a = r.sma_sb25 / pixscale; b = a * r.DISC_BA
                # PA north through east; image has north up, east LEFT -> angle = 90 + PA in matplotlib (x to the right)
                ax.add_patch(Ellipse((c, c), 2 * a, 2 * b, angle=90 + r.DISC_PA, fill=False, ec="w", lw=0.8, ls="--"))
            ax.set_title(f"{int(r.TARGETID)}\nP{int(r.PRIORITY)} z={r.Z:.4f} i={r.INCL_DEG:.0f}° e_bar={r.BAR_ELLIP:.2f}", fontsize=8)
        except Exception as exc:
            ax.text(0.5, 0.5, f"cutout failed\n{exc}"[:60], ha="center", va="center", fontsize=7, transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in np.atleast_1d(axes).ravel()[n:]:
        ax.axis("off")
    plt.tight_layout()

prio = cand[cand.PRIORITY <= 2].reset_index(drop=True)
print(len(prio), "priority targets. Links to the interactive viewer:")
display(HTML("<br>".join(f'{int(r.TARGETID)} (P{int(r.PRIORITY)}): <a href="{viewer_link(r.RA, r.DEC)}" target="_blank">{r.RA:.4f} {r.DEC:+.4f}</a>'
                         for _, r in prio.iterrows())))

In [ ]:
# show in batches of 20 so the figure stays readable; change the slice to see the rest
show_grid(prio.iloc[0:20])

In [ ]:
show_grid(prio.iloc[20:40])

In [ ]:
# ---- record your classifications here: TARGETID -> 'y' | 'w' | 'n' | 'x'   (add a note after a '#' if you like)
VET = {
    # 39627731934251234: "y  # clear bar, isolated",
}

vet_path = PROD / "vetting.csv"
old = pd.read_csv(vet_path) if vet_path.exists() else pd.DataFrame(columns=["TARGETID", "VET", "VET_NOTE"])
rows = [dict(TARGETID=int(k), VET=str(v).split("#")[0].strip(), VET_NOTE=(str(v).split("#", 1)[1].strip() if "#" in str(v) else ""))
        for k, v in VET.items()]
vet = pd.concat([old[~old.TARGETID.isin([r["TARGETID"] for r in rows])], pd.DataFrame(rows)], ignore_index=True)
vet.to_csv(vet_path, index=False)
print(f"{len(vet)} classifications saved to {vet_path}:", vet.VET.value_counts().to_dict())
cand = cand.drop(columns=[c for c in ("VET", "VET_NOTE") if c in cand]).merge(vet, on="TARGETID", how="left")
print("still to vet (priority 1-2):", int(((cand.PRIORITY <= 2) & cand.VET.isna()).sum()))

## 3. MUSE coverage from the ESO archive (step 3)

Cone search of 30" around every candidate in `ivoa.ObsCore` (reduced cubes)
and `dbo.raw` (raw science frames). Column names follow the ESO TAP
documentation; if a query fails, run the schema cell and adjust the two
query strings in `bartw/muse.py`.

In [ ]:
from bartw.muse import query_muse, summarize_coverage, print_schema
# print_schema(cfg["muse"]["tap_url"], "ivoa.ObsCore")   # uncomment to list the live column names
# print_schema(cfg["muse"]["tap_url"], "dbo.raw")

In [ ]:
hits_path = data_dir("interim") / "03_muse_hits.parquet"
targets = cand if "VET" not in cand or cand.VET.isna().all() else cand[~cand.VET.isin(["n", "x"])]
print(f"querying the ESO archive for {len(targets)} galaxies (vetted 'n'/'x' excluded) ...")
hits = query_muse(targets.RA.to_numpy(), targets.DEC.to_numpy(), cfg["muse"])
if hits.attrs.get("n_failed_queries", 0) >= len(targets):
    raise RuntimeError("every ESO query failed: no network access or the TAP service is down; nothing written")
hits.to_parquet(hits_path)
cov = summarize_coverage(targets.reset_index(drop=True), hits, cfg["muse"])
print("with reduced MUSE cubes:", int((cov.N_MUSE_CUBES > 0).sum()),
      "| public:", int(cov.MUSE_PUBLIC.sum()), "| deep enough:", int((cov.MUSE_DEEP_ENOUGH & (cov.N_MUSE_CUBES > 0)).sum()),
      "| raw only:", int(cov.MUSE_RAW_ONLY.sum()))
cov.to_csv(PROD / "sample_with_muse.csv", index=False)
cols = ["TARGETID", "sga_galaxy", "RA", "DEC", "Z", "logmstar_h0678", "INCL_DEG", "BAR_ELLIP", "PRIORITY",
        "N_MUSE_CUBES", "MUSE_EXPTIME_S", "MUSE_PUBLIC", "MUSE_RAW_ONLY", "MUSE_PROG_IDS"]
display(cov[(cov.N_MUSE_CUBES > 0) | cov.MUSE_RAW_ONLY][cols].sort_values("PRIORITY"))

In [ ]:
# Full list of archive products for one galaxy (dp_id is what you download from archive.eso.org)
if len(hits):
    tid = cov[cov.N_MUSE_CUBES > 0].TARGETID.iloc[0] if (cov.N_MUSE_CUBES > 0).any() else None
    if tid is not None:
        i = int(np.where(targets.TARGETID.to_numpy() == tid)[0][0])
        display(hits[hits.idx == i])

## 4. Optional: Galaxy Zoo DESI bars for the non-SGA dwarfs (step 2b)

94% of the star-forming dwarfs are too small for the SGA, so the bar catalogue
cannot see them. Galaxy Zoo DESI (Zenodo 8360385, `gz_desi_deep_learning_catalog_friendly.parquet`,
~2 GB) can. Put the file in `data/raw/` and run this section, then re-run
`scripts/02_crossmatch_bars.py` for the merged candidate list.

In [ ]:
from bartw.morphology import read_gz_desi, crossmatch, select_barred
gz_path = data_dir("raw") / "gz_desi_deep_learning_catalog_friendly.parquet"
if gz_path.exists():
    sf = pd.read_parquet(data_dir("interim") / "01_sf_dwarfs.parquet")
    gz = read_gz_desi(gz_path)
    m = crossmatch(sf, gz, cfg["galaxy_zoo_desi"]["match_radius_arcsec"])
    barred_gz, steps = select_barred(m, cfg["galaxy_zoo_desi"])
    print(steps)
    print("GZ-barred dwarfs at z<0.012:", int((barred_gz.Z < 0.012).sum()))
    display(barred_gz[["TARGETID", "RA", "DEC", "Z", "logmstar_h0678", "INCL_DEG", "GZ_BAR_FRACTION", "BAR_CLASS"]].head(15))
else:
    print("GZ DESI catalogue not found at", gz_path, "- see scripts/download_data.sh")

## 5. Disc and bar geometry, TW suitability (step 4)

For the MUSE-covered (or all vetted 'y'/'w') galaxies: r-band Legacy Surveys
cutout, isophote fit, disc PA/ellipticity from the outer isophotes, bar PA
and semi-major axis from the ellipticity peak. TW needs 25 < i < 70 deg and
10 < |PA_bar - PA_disc| < 80 deg. Inspect the profiles: the bar should
appear as an ellipticity bump with constant PA, followed by a drop towards
the disc ellipticity and a PA change.

In [ ]:
from bartw.photometry import fetch_cutout, isophote_profiles, disc_geometry, bar_geometry
from bartw.geometry import inclination_from_axis_ratio, tw_suitable, pa_difference

def geometry_for(row, size=512, show=True):
    g = cfg["geometry"]; pix = g["cutout_pixscale_arcsec"]
    img, hdr = fetch_cutout(row.RA, row.DEC, g["cutout_layer"], pix, size)
    prof = isophote_profiles(img)
    smax = np.nanmax(prof["sma"])
    pa_d, pa_d_err, eps_d, eps_d_err = disc_geometry(prof, 0.6 * smax, 0.95 * smax)
    bar = bar_geometry(prof, pa_d, eps_d)
    incl = float(inclination_from_axis_ratio(1 - eps_d, g["q0"]))
    res = dict(TARGETID=int(row.TARGETID), PA_DISC=pa_d, PA_DISC_ERR=pa_d_err, EPS_DISC=eps_d, INCL_DEG=incl,
               PA_BAR=bar["pa_bar"], A_BAR_ARCSEC=bar["a_bar"] * pix if np.isfinite(bar["a_bar"]) else np.nan,
               EPS_BAR=bar["eps_bar"], DPA_DEG=float(pa_difference(bar["pa_bar"], pa_d)) if np.isfinite(bar["pa_bar"]) else np.nan)
    if show:
        fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
        ax[0].imshow(np.arcsinh(img / np.nanstd(img)), origin="lower", cmap="gray_r"); ax[0].set_title(f"{int(row.TARGETID)}  r band")
        ax[1].plot(prof["sma"] * pix, prof["eps"], ".-"); ax[1].axhline(eps_d, color="C3", ls="--", label="disc")
        if np.isfinite(bar["a_bar"]): ax[1].axvline(bar["a_bar"] * pix, color="C2", label="a_bar")
        ax[1].set_xlabel("sma [arcsec]"); ax[1].set_ylabel("ellipticity"); ax[1].legend()
        ax[2].plot(prof["sma"] * pix, prof["pa"], ".-"); ax[2].axhline(pa_d, color="C3", ls="--")
        if np.isfinite(bar["pa_bar"]): ax[2].axhline(bar["pa_bar"], color="C2")
        ax[2].set_xlabel("sma [arcsec]"); ax[2].set_ylabel("PA [deg]"); plt.tight_layout()
    return res

muse_path = PROD / "sample_with_muse.csv"
todo = None
if muse_path.exists():
    cov = pd.read_csv(muse_path)
    todo = cov[(cov.N_MUSE_CUBES > 0) | cov.MUSE_RAW_ONLY]
    if len(todo) == 0:
        print("no MUSE coverage found in", muse_path.name, "- measuring the priority targets instead")
        todo = None
if todo is None:
    todo = cand[cand.PRIORITY <= 2] if "VET" not in cand or cand.VET.isna().all() else cand[cand.VET.isin(["y", "w"])]
print(len(todo), "galaxies to measure")

In [ ]:
geo = []
for _, r in todo.iterrows():
    try:
        geo.append(geometry_for(r, show=len(todo) <= 12))
    except Exception as exc:
        print(int(r.TARGETID), "failed:", exc)
geo = pd.DataFrame(geo)
if len(geo):
    ok, dpa, why = tw_suitable(geo.INCL_DEG, geo.PA_BAR, geo.PA_DISC, cfg["geometry"])
    geo["TW_SUITABLE"], geo["TW_REASON"] = ok, why
    out = todo.merge(geo, on="TARGETID", suffixes=("_cat", ""))
    out["A_BAR_KPC"] = out.A_BAR_ARCSEC * out.kpc_per_arcsec
    out.to_csv(PROD / "sample_geometry.csv", index=False)
    print("TW-suitable:", int(out.TW_SUITABLE.sum()), "/", len(out)); print(out.TW_REASON.value_counts().to_string())
    display(out[["TARGETID", "Z", "INCL_DEG_cat", "INCL_DEG", "PA_DISC", "PA_BAR", "DPA_DEG", "A_BAR_ARCSEC", "A_BAR_KPC", "EPS_BAR", "TW_SUITABLE"]].round(2))

## 6. What a TW measurement looks like (offline demo)

Synthetic barred disc with a known pattern speed, run through the same code
that `scripts/06_run_tw.py` uses on real kinematic maps. The right panel is
the diagnostic to look at for every real galaxy: points on a straight line
through the origin, errors dominated by the disc PA.

In [ ]:
from bartw.mock import MockParams, make_mock
from bartw.tw import measure_pattern_speed, default_slit_centers, rotation_rate
from bartw.plotting import slit_overlay, tw_diagram

p = MockParams(psf_fwhm_arcsec=0.8, flux_noise_frac=0.05, vel_noise_kms=5.0, omega_p=30.0)
flux, vel, verr, t = make_mock(p)
dpa = pa_difference(t["bar_pa_sky"], p.pa_deg)
yc = default_slit_centers(t["a_bar_sky_arcsec"], dpa, cfg["tw"]["n_slits"], cfg["tw"]["slit_width_arcsec"])
res = measure_pattern_speed(flux, vel, p.pixscale, t["x0"], t["y0"], p.pa_deg, p.incl_deg, yc, cfg["tw"]["slit_width_arcsec"],
                            x_half_length_arcsec=1.5 * t["a_bar_sky_arcsec"], vel_err=verr, distance_mpc=p.distance_mpc,
                            pa_err_deg=cfg["tw"]["pa_error_deg"], n_mc=200, rng=0)
print("input Omega_p =", p.omega_p, "km/s/kpc ->", res.summary())
rr = rotation_rate(res.omega_p, res.omega_p_err, vc=p.vflat, vc_err=5, a_bar_kpc=p.a_bar_arcsec * t["scale_kpc_arcsec"], a_bar_err_kpc=0.1, rng=0)
print(f"R = {rr['R']:.2f} (+{rr['R_err_hi']:.2f}/-{rr['R_err_lo']:.2f}) -> {rr['regime']}  p_slow={rr['p_slow']:.2f} p_fast={rr['p_fast']:.2f} p_uf={rr['p_ultrafast']:.2f}")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
slit_overlay(flux, vel, res, cfg["tw"]["slit_width_arcsec"], p.pixscale, True, axes[:2], 1.5 * t["a_bar_sky_arcsec"])
tw_diagram(res, axes[2], title="synthetic disc"); plt.tight_layout()

## 7. Hand-over to the scripts

* MUSE cubes: download the `dp_id`s listed above from https://archive.eso.org (public) and run
  `python scripts/05_stellar_kinematics.py CUBE.fits --z Z --templates spectra_emiles_9.0.npz --out data/products/kin_NAME.fits`
* TW: `python scripts/06_run_tw.py data/products/kin_NAME.fits --name NAME --pa PA_DISC --pa-err PA_DISC_ERR --incl INCL_DEG --bar-pa PA_BAR --a-bar A_BAR_ARCSEC --distance D_MPC [--vc VC --vc-err DVC]`
  with the numbers from `products/sample_geometry.csv`.
* Benchmark first on NGC 4516 / IC 3167 against Cuomo et al. (2022, 2024) before any new galaxy.